In [1]:
!pip install numpy pandas matplotlib seaborn scikit-learn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ===============================
# 1) CARICA TRAIN DATASET
# ===============================
train_dataset = pd.read_csv("hand_dataset_train.csv")
X = train_dataset.iloc[:, 1:].values
Y = train_dataset.iloc[:, 0].values

# Splitta in train e validation (20% validation)
X_train, X_val, y_train, y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

# ===============================
# 2) CARICA TEST DATASET
# ===============================
test_dataset = pd.read_csv("hand_dataset_test.csv")
X_test = test_dataset.iloc[:, 1:].values
y_test = test_dataset.iloc[:, 0].values

# ===============================
# 3) SCALING
# ===============================
scaler = StandardScaler().fit(X_train)

X_train = scaler.transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# ===============================
# 4) INFO SHAPE
# ===============================
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)


X_train: (21576, 42)
X_val: (5394, 42)
X_test: (2952, 42)


In [2]:
import numpy as np
classes = np.unique(y_train)
for cls in classes:
    X_cls_train = X_train[y_train == cls]
    print(f"Classe {cls}: {X_cls_train.shape[0]} campioni")


Classe A: 713 campioni
Classe B: 507 campioni
Classe C: 938 campioni
Classe D: 926 campioni
Classe E: 806 campioni
Classe F: 1023 campioni
Classe G: 986 campioni
Classe H: 966 campioni
Classe I: 966 campioni
Classe J: 370 campioni
Classe K: 1154 campioni
Classe L: 1021 campioni
Classe M: 732 campioni
Classe N: 724 campioni
Classe O: 703 campioni
Classe P: 841 campioni
Classe Q: 697 campioni
Classe R: 1022 campioni
Classe S: 903 campioni
Classe T: 774 campioni
Classe U: 694 campioni
Classe V: 990 campioni
Classe W: 923 campioni
Classe X: 970 campioni
Classe Y: 860 campioni
Classe Z: 367 campioni


In [3]:
import numpy as np
from sklearn.mixture import GaussianMixture

# Classi presenti
classes = np.unique(y_train)

# ===============================
# 2) STIMA PRIORS (con train+val)
# ===============================
priors = {}
for cls in classes:
    priors[cls] = np.sum(y_train == cls) / len(y_train)

print("Priors:", priors)

# Numero di componenti candidate
component_candidates = [1, 2, 3, 4, 5, 6, 8, 10]

# Dizionari per salvare i migliori risultati
best_components_bic = {}
best_components_aic = {}
best_components_ll  = {}
best_gmms_bic = {}
best_gmms_aic = {}
best_gmms_ll  = {}

for cls in classes:
    Xc_train = X_train[y_train == cls]
    Xc_val   = X_val[y_val == cls]

    # Inizializza variabili per confronto
    best_bic = np.inf
    best_aic = np.inf
    best_ll  = -np.inf
    best_gmm_bic = None
    best_gmm_aic = None
    best_gmm_ll  = None

    for k in component_candidates:
        gmm = GaussianMixture(
            n_components=k,
            covariance_type='full',
            random_state=42
        )
        gmm.fit(Xc_train)

        # Calcolo criteri
        bic = gmm.bic(Xc_val)
        aic = gmm.aic(Xc_val)
        ll  = gmm.score(Xc_val) * len(Xc_val)  # log-likelihood totale

        # Salva migliore BIC
        if bic < best_bic:
            best_bic = bic
            best_components_bic[cls] = k
            best_gmms_bic[cls] = gmm

        # Salva migliore AIC
        if aic < best_aic:
            best_aic = aic
            best_components_aic[cls] = k
            best_gmms_aic[cls] = gmm

        # Salva migliore log-likelihood pura
        if ll > best_ll:
            best_ll = ll
            best_components_ll[cls] = k
            best_gmms_ll[cls] = gmm

    print(f"Classe {cls}: BIC={best_components_bic[cls]}, AIC={best_components_aic[cls]}, LL={best_components_ll[cls]}")

# Ricombinazione train + val per allenamento finale
X_train_full = np.concatenate([X_train, X_val], axis=0)
y_train_full = np.concatenate([y_train, y_val], axis=0)

print("\nRicombinazione completata:")
print("X_train_full:", X_train_full.shape)


Priors: {'A': 0.033045977011494254, 'B': 0.02349833147942158, 'C': 0.04347423062662217, 'D': 0.042918057100482014, 'E': 0.03735632183908046, 'F': 0.04741379310344827, 'G': 0.0456989247311828, 'H': 0.044771968854282536, 'I': 0.044771968854282536, 'J': 0.017148683722654802, 'K': 0.05348535409714498, 'L': 0.04732109751575825, 'M': 0.0339265850945495, 'N': 0.033555802743789394, 'O': 0.03258249907304412, 'P': 0.038978494623655914, 'Q': 0.032304412309974044, 'R': 0.04736744530960326, 'S': 0.041852057842046715, 'T': 0.03587319243604004, 'U': 0.032165368928439006, 'V': 0.04588431590656285, 'W': 0.042779013718946976, 'X': 0.04495736002966259, 'Y': 0.03985910270671116, 'Z': 0.017009640341119764}
Classe A: BIC=1, AIC=2, LL=3
Classe B: BIC=1, AIC=2, LL=2
Classe C: BIC=1, AIC=2, LL=3
Classe D: BIC=1, AIC=2, LL=2
Classe E: BIC=1, AIC=2, LL=4
Classe F: BIC=1, AIC=2, LL=4
Classe G: BIC=1, AIC=2, LL=5
Classe H: BIC=1, AIC=2, LL=2
Classe I: BIC=1, AIC=3, LL=3
Classe J: BIC=1, AIC=1, LL=1
Classe K: BIC=1

In [4]:
# ===============================
# 5) TRAIN FINALE DEI GMM PER CLASSE
# ===============================

final_gmms_bic = {}
final_gmms_aic = {}
final_gmms_ll  = {}

for cls in classes:

    # Recupera il numero di componenti scelto da ciascun criterio
    k_bic = best_components_bic[cls]
    k_aic = best_components_aic[cls]
    k_ll  = best_components_ll[cls]

    Xc = X_train_full[y_train_full == cls]

    # ----- BIC -----
    print(f"Addestro modello finale per classe {cls} con {k_bic} componenti (BIC)...")
    gmm_bic = GaussianMixture(
        n_components=k_bic,
        covariance_type='full',
        random_state=42
    )
    gmm_bic.fit(Xc)
    final_gmms_bic[cls] = gmm_bic

    # ----- AIC -----
    print(f"Addestro modello finale per classe {cls} con {k_aic} componenti (AIC)...")
    gmm_aic = GaussianMixture(
        n_components=k_aic,
        covariance_type='full',
        random_state=42
    )
    gmm_aic.fit(Xc)
    final_gmms_aic[cls] = gmm_aic

    # ----- Log-likelihood pura -----
    print(f"Addestro modello finale per classe {cls} con {k_ll} componenti (Log-Likelihood)...")
    gmm_ll = GaussianMixture(
        n_components=k_ll,
        covariance_type='full',
        random_state=42
    )
    gmm_ll.fit(Xc)
    final_gmms_ll[cls] = gmm_ll


Addestro modello finale per classe A con 1 componenti (BIC)...
Addestro modello finale per classe A con 2 componenti (AIC)...
Addestro modello finale per classe A con 3 componenti (Log-Likelihood)...
Addestro modello finale per classe B con 1 componenti (BIC)...
Addestro modello finale per classe B con 2 componenti (AIC)...
Addestro modello finale per classe B con 2 componenti (Log-Likelihood)...
Addestro modello finale per classe C con 1 componenti (BIC)...
Addestro modello finale per classe C con 2 componenti (AIC)...
Addestro modello finale per classe C con 3 componenti (Log-Likelihood)...
Addestro modello finale per classe D con 1 componenti (BIC)...
Addestro modello finale per classe D con 2 componenti (AIC)...
Addestro modello finale per classe D con 2 componenti (Log-Likelihood)...
Addestro modello finale per classe E con 1 componenti (BIC)...
Addestro modello finale per classe E con 2 componenti (AIC)...
Addestro modello finale per classe E con 4 componenti (Log-Likelihood)...


In [5]:
from sklearn.metrics import accuracy_score

# ===============================
# 6) PREDIZIONE SUL TEST
# ===============================

def predict_gmms(X, gmms, priors, classes):
    y_pred = []
    for x in X:
        class_scores = {}
        for cls in classes:
            log_like = gmms[cls].score(x.reshape(1, -1))
            log_post = log_like + np.log(priors[cls])
            class_scores[cls] = log_post
        y_pred.append(max(class_scores, key=class_scores.get))
    return np.array(y_pred)

# Predizione con BIC
y_pred_bic = predict_gmms(X_test, final_gmms_bic, priors, classes)
acc_bic = accuracy_score(y_test, y_pred_bic)
print(f"Accuracy sul test (BIC): {acc_bic*100:.2f}%")

# Predizione con AIC
y_pred_aic = predict_gmms(X_test, final_gmms_aic, priors, classes)
acc_aic = accuracy_score(y_test, y_pred_aic)
print(f"Accuracy sul test (AIC): {acc_aic*100:.2f}%")

# Predizione con Log-Likelihood pura
y_pred_ll = predict_gmms(X_test, final_gmms_ll, priors, classes)
acc_ll = accuracy_score(y_test, y_pred_ll)
print(f"Accuracy sul test (Log-Likelihood): {acc_ll*100:.2f}%")


Accuracy sul test (BIC): 77.57%
Accuracy sul test (AIC): 84.08%
Accuracy sul test (Log-Likelihood): 77.30%
